# Comparison

# Theme

In [58]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import os, json, subprocess, re, sys, tempfile, time
from pathlib import Path
from collections import Counter

COLORS = {"bg":"#F5F5F5","teal":"#76ABAE","dark":"#303841","orange":"#FF5722","white":"#FFFFFF"}

# Setup

In [59]:
from getpass import getpass
os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")
print("Key loaded")

Key loaded


## Solc Version Resolver

Same logic as `01_data_engineering.ipynb` — maps pragma to pinned solc version and finds the binary in `.solc-select/artifacts/`.

In [60]:
PRAGMA_RE = re.compile(r"pragma\s+solidity\s+([^;]+);")
SOLC_PINS = {"0.4":"0.4.26","0.5":"0.5.17","0.6":"0.6.12","0.7":"0.7.6","0.8":"0.8.20"}

VENV_ROOT = os.path.dirname(os.path.dirname(sys.executable))
HOME = os.path.expanduser("~")

def find_solc_bin(version):
    """Find solc binary — checks venv first, then home .solc-select."""
    for base in [
        os.path.join(VENV_ROOT, ".solc-select", "artifacts"),
        os.path.join(HOME, ".solc-select", "artifacts"),
    ]:
        path = os.path.join(base, f"solc-{version}", f"solc-{version}")
        if os.path.isfile(path):
            return path
    return None

def resolve_solc_for_file(filepath):
    """Read pragma from file and return (solc_path, series)."""
    text = open(filepath, errors="ignore").read()
    m = PRAGMA_RE.search(text)
    if not m:
        return find_solc_bin(SOLC_PINS["0.8"]), "0.8"
    pragma = m.group(1)
    vm = re.search(r"(\d+)\.(\d+)", pragma)
    if not vm:
        return find_solc_bin(SOLC_PINS["0.8"]), "0.8"
    series = vm.group(0)
    pin = SOLC_PINS.get(series)
    if pin:
        path = find_solc_bin(pin)
        if path:
            return path, series
    return find_solc_bin(SOLC_PINS["0.8"]), "0.8"

# Verify all solc versions are available
for series, pin in sorted(SOLC_PINS.items()):
    path = find_solc_bin(pin)
    status = f"✓ {path}" if path else "✗ NOT FOUND"
    print(f"solc {pin}: {status}")

solc 0.4.26: ✓ /Users/mk/Desktop/mk/AuditMind/venv/.solc-select/artifacts/solc-0.4.26/solc-0.4.26
solc 0.5.17: ✓ /Users/mk/Desktop/mk/AuditMind/venv/.solc-select/artifacts/solc-0.5.17/solc-0.5.17
solc 0.6.12: ✓ /Users/mk/Desktop/mk/AuditMind/venv/.solc-select/artifacts/solc-0.6.12/solc-0.6.12
solc 0.7.6: ✓ /Users/mk/Desktop/mk/AuditMind/venv/.solc-select/artifacts/solc-0.7.6/solc-0.7.6
solc 0.8.20: ✓ /Users/mk/.solc-select/artifacts/solc-0.8.20/solc-0.8.20


## Re-Pull Repos for Import Resolution

The 723 files are already compile-verified from `01_data_engineering.ipynb`.  
We re-pull repos here **only** so Slither can resolve `import` statements at analysis time.

In [61]:
# ── Clean-dataset repos (same as notebook 01) ──
CLEAN_REPOS = [
    ("OpenZeppelin",    "openzeppelin-contracts",              "master"),
    ("OpenZeppelin",    "openzeppelin-contracts-upgradeable",  "master"),
    ("transmissions11", "solmate",                             "main"),
    ("aave",            "aave-v3-core",                       "master"),
    ("compound-finance","comet",                               "main"),
    ("ProjectOpenSea",  "seaport",                             "main"),
    ("Uniswap",         "v4-core",                             "main"),
    ("morpho-org",      "morpho-blue",                         "main"),
    ("1inch",           "limit-order-protocol",                "master"),
    ("yearn",           "yearn-vaults-v3",                     "master"),
]

for org, repo, branch in CLEAN_REPOS:
    target = f"{repo}-{branch}"
    if os.path.isdir(target):
        print(f"  {target} ✓ exists")
        continue
    print(f"  Pulling {org}/{repo} ({branch})...", end=" ")
    rc = subprocess.run(
        ["curl", "-sL", "--fail", "--connect-timeout", "30", "--max-time", "180",
         f"https://codeload.github.com/{org}/{repo}/tar.gz/refs/heads/{branch}",
         "-o", f"{repo}.tar.gz"]).returncode
    if rc == 0:
        subprocess.run(["tar", "-xzf", f"{repo}.tar.gz"], check=True)
        os.remove(f"{repo}.tar.gz")
        print("✓")
    else:
        print("✗ FAILED")

# ── DAppSCAN sparse checkout (same as notebook 01) ──
if not os.path.isdir("dappscan_sparse"):
    print("  Cloning DAppSCAN (sparse)...", end=" ")
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
        "https://github.com/InPlusLab/DAppSCAN.git", "dappscan_sparse"], check=True)
    # Build the checkout list from our manifest's repo_roots
    dappscan_manifest = json.load(open("data/manifests/dappscan.json"))
    project_dirs = set()
    for r in dappscan_manifest:
        rr = r.get("repo_root", "")
        if rr.startswith("dappscan_sparse/"):
            # Get the DAppSCAN-source/contracts/<project> prefix
            parts = rr.replace("dappscan_sparse/", "").split("/")
            if len(parts) >= 3:
                project_dirs.add("/".join(parts[:3]))
    checkout_file = "dappscan_sparse/checkout_list.txt"
    with open(checkout_file, "w") as f:
        for d in sorted(project_dirs):
            f.write(d + "\n")
        f.write("DAppSCAN-source/SWCsource\n")
    subprocess.run(["git", "sparse-checkout", "set", "--stdin"],
                   stdin=open(checkout_file), cwd="dappscan_sparse", check=True)
    print("✓")
else:
    print("  dappscan_sparse ✓ exists")

# ── OpenZeppelin npm packages ──
INCLUDE_OZ = os.path.abspath("node_modules_oz/node_modules")
if not os.path.isdir(os.path.join(INCLUDE_OZ, "@openzeppelin")):
    os.makedirs("node_modules_oz", exist_ok=True)
    subprocess.run(["npm", "install", "@openzeppelin/contracts@5.0.2",
                    "@openzeppelin/contracts-upgradeable@5.0.2",
                    "--silent", "--no-fund", "--no-audit"], cwd="node_modules_oz", check=True)
    print("  @openzeppelin npm packages ✓ installed")
else:
    print("  @openzeppelin npm packages ✓ exist")

print(f"\n✓ All repos and dependencies ready")

  openzeppelin-contracts-master ✓ exists
  openzeppelin-contracts-upgradeable-master ✓ exists
  solmate-main ✓ exists
  aave-v3-core-master ✓ exists
  comet-main ✓ exists
  seaport-main ✓ exists
  v4-core-main ✓ exists
  morpho-blue-main ✓ exists
  limit-order-protocol-master ✓ exists
  yearn-vaults-v3-master ✓ exists
  Cloning DAppSCAN (sparse)... 

Cloning into 'dappscan_sparse'...
Updating files: 100% (4/4), done.


✓
  @openzeppelin npm packages ✓ exist

✓ All repos and dependencies ready


## Load Manifest — the exact 723 files

In [62]:
manifest = json.load(open("data/manifests/stage1_manifest.json"))
assert len(manifest) == 723, f"Expected 723 files, got {len(manifest)}"

def build_slither_path(rec):
    """Reconstruct the ORIGINAL repo path so Slither can resolve imports.
    
    - smartbugs / swc: self-contained, use path as-is (data/smartbugs/...)
    - dappscan: reconstruct from flat name + repo_root → dappscan_sparse/...
    - clean: reconstruct from flat name + repo_root → <repo>-<branch>/...
    """
    source = rec.get("source", "")
    
    if source in ("smartbugs", "swc"):
        # Self-contained files, no imports — use existing path
        return rec["path"]
    
    # For dappscan and clean: reconstruct original path from flat name
    flat_name = os.path.basename(rec["path"])
    repo_root = rec.get("repo_root", "")
    prefix = repo_root.replace("/", "__") + "__"
    if flat_name.startswith(prefix):
        rel = flat_name[len(prefix):].replace("__", "/")
        return os.path.join(repo_root, rel)
    
    # Fallback: use existing path
    return rec["path"]

# Build paths and verify ALL 723 exist
found = 0
missing_list = []
for rec in manifest:
    rec["slither_path"] = build_slither_path(rec)
    if os.path.exists(rec["slither_path"]):
        found += 1
    else:
        missing_list.append(rec)

print(f"Total: {len(manifest)} | Found: {found} | Missing: {len(missing_list)}")

# For any missing files (repos may have changed), fall back to the flat copy
for rec in missing_list:
    if os.path.exists(rec["path"]):
        rec["slither_path"] = rec["path"]
        print(f"  Fallback to flat copy: {os.path.basename(rec['path'])}")

# Final check
all_exist = all(os.path.exists(r["slither_path"]) for r in manifest)
print(f"\nAll 723 files accessible: {all_exist}")
assert all_exist, "Some files are missing!"

# Breakdown
for src in ["smartbugs", "swc", "dappscan", "clean"]:
    entries = [r for r in manifest if r.get("source") == src]
    vuln = sum(1 for r in entries if r.get("label_type") == "vulnerable")
    clean = len(entries) - vuln
    print(f"  {src:10s}: {len(entries):3d} files  (vuln={vuln}, clean={clean})")

Total: 723 | Found: 723 | Missing: 0

All 723 files accessible: True
  smartbugs : 116 files  (vuln=116, clean=0)
  swc       :  13 files  (vuln=13, clean=0)
  dappscan  :  83 files  (vuln=83, clean=0)
  clean     : 511 files  (vuln=0, clean=511)


# Finding Class & Slither

In [63]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class Finding:
    vuln_type: str
    severity: str
    line_number: int | None
    description: str
    source: str = "slither"

In [64]:
_DETECTOR_TO_CATEGORY = {
    "reentrancy-eth": "reentrancy", "reentrancy-no-eth": "reentrancy",
    "reentrancy-benign": "reentrancy", "reentrancy-events": "reentrancy",
    "unprotected-upgrade": "access_control", "arbitrary-send-eth": "access_control",
    "suicidal": "access_control", "tx-origin": "access_control",
    "timestamp": "time_manipulation",
    "low-level-calls": "unchecked_low_level_calls",
    "unchecked-transfer": "unchecked_low_level_calls",
    "unchecked-lowlevel": "unchecked_low_level_calls",
    "calls-loop": "denial_of_service", "costly-loop": "denial_of_service",
}
_IMPACT_TO_SEVERITY = {"High":"high","Medium":"medium","Low":"low","Informational":"low","Optimization":"low"}

def _parse_slither_json(raw_json):
    findings = []
    for det in raw_json.get("results", {}).get("detectors", []):
        cat = _DETECTOR_TO_CATEGORY.get(det.get("check", ""), det.get("check", "unknown"))
        sev = _IMPACT_TO_SEVERITY.get(det.get("impact", ""), "low")
        desc = det.get("description", "").strip()
        line = None
        elems = det.get("elements", [])
        if elems:
            lines = elems[0].get("source_mapping", {}).get("lines", [])
            if lines: line = lines[0]
        findings.append(Finding(cat, sev, line, desc))
    return findings

SLITHER_BIN = os.path.join(VENV_ROOT, "bin", "slither")

def run_slither(rec, timeout_s=60):
    """Run Slither on a manifest record using the correct solc + import paths.
    
    Mirrors the exact compile logic from 01_data_engineering.ipynb:
    - smartbugs/swc:  solc --bin <file>
    - dappscan:       solc --bin <file> --base-path <repo_root>   (cwd=repo_root)
    - clean:          solc --bin <file> --base-path <repo_root> --include-path <OZ>  (cwd=repo_root)
    """
    filepath = rec["slither_path"]
    if not os.path.exists(filepath):
        return [Finding("analysis_error","high",None,f"File not found: {filepath}")]
    
    solc_path, series = resolve_solc_for_file(filepath)
    if not solc_path:
        return [Finding("analysis_error","high",None,"No solc binary found")]
    
    abs_file = os.path.abspath(filepath)
    source = rec.get("source", "")
    
    # Write JSON to temp file (slither --json - is unreliable)
    tmp = tempfile.mktemp(suffix=".json")
    cmd = [SLITHER_BIN, abs_file, "--json", tmp, "--solc", solc_path]
    cwd = None  # default
    
    # Build --solc-args exactly like notebook 01's compile commands
    solc_extra = []
    if source == "dappscan":
        repo_root = rec.get("repo_root", os.path.dirname(filepath))
        abs_root = os.path.abspath(repo_root)
        if os.path.isdir(abs_root) and series in ("0.6", "0.7", "0.8"):
            solc_extra += ["--base-path", abs_root]
            cwd = abs_root
        if series == "0.8":
            solc_extra += ["--include-path", INCLUDE_OZ]
    elif source == "clean":
        repo_root = rec.get("repo_root", "")
        abs_root = os.path.abspath(repo_root)
        if os.path.isdir(abs_root):
            solc_extra += ["--base-path", abs_root]
            solc_extra += ["--include-path", INCLUDE_OZ]
            cwd = abs_root
    
    if solc_extra:
        cmd += ["--solc-args", " ".join(solc_extra)]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True,
                               timeout=timeout_s, cwd=cwd)
    except subprocess.TimeoutExpired:
        if os.path.exists(tmp): os.unlink(tmp)
        return [Finding("analysis_error","high",None,"Slither timed out")]
    except Exception as e:
        if os.path.exists(tmp): os.unlink(tmp)
        return [Finding("analysis_error","high",None,f"Slither error: {e}")]
    
    # Read JSON from temp file
    if not os.path.exists(tmp) or os.path.getsize(tmp) == 0:
        if os.path.exists(tmp): os.unlink(tmp)
        # Slither crashed but might have still produced text-mode output
        snippet = (result.stderr or result.stdout or "")[:300]
        return [Finding("analysis_error","high",None,f"No JSON: {snippet}")]
    
    try:
        raw_json = json.load(open(tmp))
    except json.JSONDecodeError as e:
        os.unlink(tmp)
        return [Finding("analysis_error","high",None,f"JSON parse: {e}")]
    finally:
        if os.path.exists(tmp): os.unlink(tmp)
    
    if not raw_json.get("success", True):
        return [Finding("analysis_error","high",None,f"Slither: {raw_json.get('error','unknown')}")]
    
    return _parse_slither_json(raw_json)

## Run Slither on All 723 Files

In [65]:
print(f"Running Slither on all {len(manifest)} files...")
t0 = time.time()

for i, rec in enumerate(manifest):
    rec["slither_findings"] = run_slither(rec)
    if (i+1) % 50 == 0:
        ok = sum(1 for r in manifest[:i+1]
                 if not any(f.vuln_type == "analysis_error" for f in r["slither_findings"]))
        elapsed = time.time() - t0
        print(f"  {i+1}/{len(manifest)} done  ({ok} analyzed, {elapsed:.0f}s)")

elapsed = time.time() - t0
slither_ok = sum(1 for r in manifest if not any(f.vuln_type == "analysis_error" for f in r["slither_findings"]))
slither_err = len(manifest) - slither_ok
print(f"\n✓ Slither complete in {elapsed:.0f}s")
print(f"  Analyzed: {slither_ok}/{len(manifest)}")
print(f"  Errors:   {slither_err}/{len(manifest)}")

# Breakdown by source
for src in ["smartbugs", "swc", "dappscan", "clean"]:
    entries = [r for r in manifest if r.get("source") == src]
    ok = sum(1 for r in entries if not any(f.vuln_type == "analysis_error" for f in r["slither_findings"]))
    print(f"  {src:10s}: {ok}/{len(entries)} analyzed")

Running Slither on all 723 files...
  50/723 done  (48 analyzed, 23s)
  100/723 done  (97 analyzed, 41s)
  150/723 done  (142 analyzed, 77s)
  200/723 done  (183 analyzed, 106s)
  250/723 done  (192 analyzed, 122s)
  300/723 done  (192 analyzed, 135s)
  350/723 done  (192 analyzed, 147s)
  400/723 done  (192 analyzed, 159s)
  450/723 done  (192 analyzed, 172s)
  500/723 done  (194 analyzed, 185s)
  550/723 done  (244 analyzed, 213s)
  600/723 done  (292 analyzed, 237s)
  650/723 done  (292 analyzed, 250s)
  700/723 done  (292 analyzed, 262s)

✓ Slither complete in 268s
  Analyzed: 301/723
  Errors:   422/723
  smartbugs : 113/116 analyzed
  swc       : 13/13 analyzed
  dappscan  : 66/83 analyzed
  clean     : 109/511 analyzed


# Run Haiku

In [ ]:
import anthropic
client = anthropic.Anthropic()

JUDGE_PROMPT = """You are checking a Solidity contract for well-defined, pattern-matchable vulnerabilities ONLY. Do NOT reason about business logic intent.

Check specifically for:
- Reentrancy (external call before state update)
- Missing access control on sensitive functions
- Unbounded loops (potential DoS via gas exhaustion)
- Timestamp manipulation risk
- Unchecked low-level calls

Contract:
```solidity
{contract_source}
```

Respond ONLY with JSON: {{"findings": [{{"vuln_type": "<reentrancy|access_control|denial_of_service|time_manipulation|unchecked_low_level_calls>", "severity": "<high|medium|low>", "line_number": <int or null>, "description": "<one sentence>"}}]}}
If none found: {{"findings": []}}"""

def judge_contract(contract_source):
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=800, temperature=0,
        messages=[{"role":"user","content":JUDGE_PROMPT.format(contract_source=contract_source)}])
    text = resp.content[0].text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"): text = text[4:]
        text = text.strip()
    try:
        result = json.loads(text)
        return [Finding(f["vuln_type"], f.get("severity","low"), f.get("line_number"),
                         f.get("description",""), source="haiku_judge")
                for f in result.get("findings", [])]
    except (json.JSONDecodeError, KeyError) as e:
        return [Finding("analysis_error","high",None,f"Parse failed: {e}", source="haiku_judge")]

## Run Haiku on All 723 Files

In [ ]:
print(f"Running Haiku judge on all {len(manifest)} files...")
t0 = time.time()

for i, rec in enumerate(manifest):
    # Read from stage1 (comment-stripped) version
    read_path = rec.get("stage1_path", rec["slither_path"])
    if not os.path.exists(read_path):
        read_path = rec["slither_path"]
    try:
        source_code = open(read_path, errors="ignore").read()
    except Exception as e:
        rec["haiku_findings"] = [Finding("analysis_error","high",None,f"Read failed: {e}", source="haiku_judge")]
        continue
    
    try:
        rec["haiku_findings"] = judge_contract(source_code)
    except Exception as e:
        rec["haiku_findings"] = [Finding("analysis_error","high",None,f"API error: {e}", source="haiku_judge")]
    
    if (i+1) % 50 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{len(manifest)} done ({elapsed:.0f}s)")
    time.sleep(0.1)

elapsed = time.time() - t0
haiku_ok = sum(1 for r in manifest if not any(f.vuln_type == "analysis_error" for f in r.get("haiku_findings", [])))
print(f"\n✓ Haiku complete in {elapsed:.0f}s")
print(f"  Analyzed: {haiku_ok}/{len(manifest)}")

# Save Results

In [ ]:
os.makedirs("data/stage2_results", exist_ok=True)

def findings_to_dicts(findings):
    return [{"vuln_type": f.vuln_type, "severity": f.severity,
             "line_number": f.line_number, "description": f.description,
             "source": f.source} for f in findings]

results = []
for rec in manifest:
    results.append({
        "path": rec["path"],
        "slither_path": rec.get("slither_path"),
        "stage1_path": rec.get("stage1_path"),
        "source": rec.get("source"),
        "label_type": rec.get("label_type"),
        "category": rec.get("category", rec.get("categories")),
        "slither_findings": findings_to_dicts(rec.get("slither_findings", [])),
        "haiku_findings": findings_to_dicts(rec.get("haiku_findings", [])),
    })

json.dump(results, open("data/stage2_results/comparison_results.json", "w"), indent=2)
print(f"✓ Saved {len(results)} results to data/stage2_results/comparison_results.json")

# Diagram

In [ ]:
MAIN_X, MAIN_W = 0.0, 6.0
DROP_X, DROP_W = 6.8, 5.2
GAP = 0.55
LINE = 0.34

def _h(detail):
    n = len(detail) if detail else 0
    return 0.62 + (0.30 + LINE * n if n else 0)

def draw_flowchart(title, stages):
    rows, y = [], 0.0
    for s in stages:
        if s["type"] == "drop":
            rows.append((s, None)); continue
        h = _h(s.get("detail"))
        rows.append((s, h)); y += h + GAP
    total = y + 1.0

    fig, ax = plt.subplots(figsize=(11, total * 0.78))
    fig.patch.set_facecolor(COLORS["bg"]); ax.set_facecolor(COLORS["bg"])
    ax.set_xlim(-0.4, 12.4); ax.set_ylim(0, total); ax.axis("off")

    cy = total - 0.7
    prev_bottom = None

    for s, h in rows:
        if s["type"] == "drop":
            d = s.get("detail") or []
            dh = 0.62 + 0.30 + LINE * len(d)
            top = prev_bottom - 0.10
            by = top - dh
            ax.add_patch(FancyBboxPatch((DROP_X, by), DROP_W, dh,
                boxstyle="round,pad=0.02,rounding_size=0.12",
                fc=COLORS["white"], ec=COLORS["orange"], lw=1.4))
            ax.text(DROP_X+0.28, top-0.36, s["label"], fontsize=10.5,
                    fontweight="bold", color=COLORS["orange"], va="center")
            ty = top - 0.36 - 0.42
            for ln in d:
                ax.text(DROP_X+0.28, ty, ln, fontsize=9, color=COLORS["dark"], va="center")
                ty -= LINE
            ymid = by + dh/2
            ax.annotate("", xy=(DROP_X-0.04, ymid), xytext=(MAIN_W+0.04, ymid),
                        arrowprops=dict(arrowstyle="-|>", color=COLORS["orange"],
                                        lw=1.4, shrinkA=0, shrinkB=0))
            continue

        final = s["type"] == "final"
        by = cy - h
        fc = COLORS["teal"] if final else COLORS["white"]
        ec = COLORS["teal"] if final else COLORS["dark"]
        tc = COLORS["white"] if final else COLORS["dark"]
        ax.add_patch(FancyBboxPatch((MAIN_X, by), MAIN_W, h,
            boxstyle="round,pad=0.02,rounding_size=0.12", fc=fc, ec=ec, lw=1.6))
        ax.text(MAIN_X+0.30, cy-0.36, s["label"], fontsize=11.5,
                fontweight="bold", color=tc, va="center")
        d = s.get("detail")
        if d:
            ty = cy - 0.36 - 0.42
            for ln in (d if isinstance(d, list) else [d]):
                ax.text(MAIN_X+0.30, ty, ln, fontsize=9, color=tc, va="center")
                ty -= LINE
        if prev_bottom is not None:
            ax.annotate("", xy=(MAIN_W/2, cy), xytext=(MAIN_W/2, prev_bottom),
                        arrowprops=dict(arrowstyle="-|>", color=COLORS["dark"],
                                        lw=1.5, shrinkA=0, shrinkB=0))
        prev_bottom = by
        cy = by - GAP

    ax.text(-0.4, total-0.18, title, fontsize=14, fontweight="bold",
            color=COLORS["dark"], va="center")
    plt.subplots_adjust(left=0.03, right=0.99, top=0.97, bottom=0.02)
    plt.show()

In [1]:
import os, sys, json, time
from pathlib import Path
from collections import Counter, defaultdict
from getpass import getpass
import anthropic

os.chdir("/Users/mk/Desktop/mk/AuditMind")

STAGE2 = Path("data/stage2_results")
STAGE3 = Path("data/stage3_scored")

CATEGORIES = ["reentrancy", "access_control", "unchecked_low_level_calls",
              "denial_of_service", "time_manipulation"]

manifest        = json.load(open("data/manifests/stage1_manifest.json"))
slither_results = json.load(open(STAGE2 / "slither_results.json"))
slither_ok      = {r["path"] for r in slither_results if r["status"] == "ok"}
targets         = [r for r in manifest if r["path"] in slither_ok]

def numbered(p):
    return "\n".join(f"{i}: {l}" for i, l in
                     enumerate(open(p, errors="ignore").read().split("\n"), 1))

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")
client = anthropic.Anthropic()
HAIKU_MODEL = "claude-haiku-4-5-20251001"

print(f"{len(manifest)} in manifest, {len(targets)} targets")
print("existing results:", sorted(os.listdir(STAGE2)))

723 in manifest, 678 targets
existing results: ['haiku_batch_meta.json', 'haiku_results.json', 'slither_failures.json', 'slither_results.json']


In [2]:
FORCED_PROMPT = """You are a Solidity security analyzer. Pick the single vulnerability category
that best describes this contract's most serious flaw, or "none" if none of them apply.

The contract's pragma is shown near the top. Judge exploitability against that version. Some
patterns that are vulnerable in 0.4.x are safe by default in 0.8.x.

CATEGORIES

reentrancy
  An external call transfers control to another contract before the caller's state is updated,
  allowing the callee to re-enter and exploit stale state.
  Not this: .transfer() or .send() with the 2300 gas stipend, or a function already protected by
  a reentrancy guard modifier.

access_control
  A function that changes ownership, moves funds, or alters critical state is callable by an
  unauthorized party. Includes missing modifiers, wrong visibility, unprotected initializers,
  tx.origin authentication, and unprotected selfdestruct or delegatecall.
  Not this: a view or pure function without a modifier, or a function that is intentionally
  public and changes only the caller's own state.

unchecked_low_level_calls
  The return value of .call, .delegatecall, .staticcall, or .send is not checked, so a failed
  call proceeds silently as if it succeeded.
  Not this: a call whose return value is checked by require, assert, or an if that reverts.

denial_of_service
  Execution can be blocked or made unaffordable. Includes unbounded loops over arrays that grow
  with user input, and functions whose progress depends on an external call a participant can
  force to fail.
  Not this: a loop bounded by a fixed constant, or one over an array only the owner can extend.

time_manipulation
  block.timestamp or block.number is used for logic a miner can profitably influence, such as
  randomness, deadlines, payout eligibility, or lock expiry.
  Not this: block.timestamp used for an event log, or for a deadline measured in days where a
  few seconds of miner drift is irrelevant.

none
  None of the five applies. Choose this for contracts that are safe with respect to these five
  categories, even if they have other problems.

If more than one applies, pick the most serious one. Do not hedge.

OUTPUT

Return a single JSON object and nothing else. No prose, no markdown fences.

{"category": "reentrancy", "lines": [42]}

"category" is exactly one of: reentrancy, access_control, unchecked_low_level_calls,
denial_of_service, time_manipulation, none.
"lines" holds the line where it is triggered, and is empty when category is "none"."""

open("data/manifests/haiku_forced_prompt.txt", "w").write(FORCED_PROMPT)
print(len(FORCED_PROMPT), "chars")

2502 chars


In [3]:
forced_reqs = [{
    "custom_id": f"f{i:04d}c0",
    "params": {"model": HAIKU_MODEL, "max_tokens": 200, "temperature": 0,
               "system": FORCED_PROMPT,
               "messages": [{"role": "user", "content": numbered(r["stage1_path"])}]},
} for i, r in enumerate(targets)]

fb = client.messages.batches.create(requests=forced_reqs)
json.dump({"batch_id": fb.id, "mode": "forced_choice"},
          open(STAGE2 / "haiku_forced_batch_meta.json", "w"), indent=2)
print(fb.id, len(forced_reqs), "requests")

msgbatch_01QTihp4ecdABnhgJCaVmLKA 678 requests


In [4]:
bid = json.load(open(STAGE2 / "haiku_forced_batch_meta.json"))["batch_id"]
while True:
    b = client.messages.batches.retrieve(bid)
    print(b.processing_status, b.request_counts)
    if b.processing_status == "ended": break
    time.sleep(30)

in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, suc

KeyboardInterrupt: 

In [5]:
bid = json.load(open(STAGE2 / "haiku_forced_batch_meta.json"))["batch_id"]
while True:
    b = client.messages.batches.retrieve(bid)
    print(b.processing_status, b.request_counts)
    if b.processing_status == "ended": break
    time.sleep(30)

in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=678, suc

In [6]:
def parse_forced(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("```")[1]
        t = t[4:] if t.startswith("json") else t
        t = t.strip()
    try: o = json.loads(t)
    except json.JSONDecodeError: return None
    c = o.get("category")
    if c not in CATEGORIES + ["none"]: return None
    ls = o.get("lines") or []
    return {"category": c, "lines": [int(x) for x in ls if isinstance(x, (int, float))]}

got = {}
for e in client.messages.batches.results(bid):
    if e.result.type != "succeeded": continue
    m = e.result.message
    got[int(e.custom_id[1:5])] = {"text": m.content[0].text,
                                  "input_tokens": m.usage.input_tokens,
                                  "output_tokens": m.usage.output_tokens}

forced_results = []
for i, r in enumerate(targets):
    g = got.get(i)
    v = parse_forced(g["text"]) if g else None
    forced_results.append({
        "path": r["path"], "source": r["source"], "label_type": r["label_type"],
        "status": "ok" if v else ("parse_error" if g else "missing"),
        "predicted": v["category"] if v else None,
        "lines": v["lines"] if v else [],
        "raw_text": g["text"] if g else "",
        "input_tokens": g["input_tokens"] if g else 0,
        "output_tokens": g["output_tokens"] if g else 0,
    })

json.dump(forced_results, open(STAGE2 / "haiku_forced_results.json", "w"))
ti = sum(r["input_tokens"] for r in forced_results)
to = sum(r["output_tokens"] for r in forced_results)
print(Counter(r["status"] for r in forced_results))
print(f"tokens in {ti:,} out {to:,} | ${ti/1e6*.5 + to/1e6*2.5:.2f} batch")
print("predictions:", Counter(r["predicted"] for r in forced_results))

Counter({'ok': 631, 'parse_error': 47})
tokens in 1,482,960 out 97,055 | $0.98 batch
predictions: Counter({'none': 406, 'access_control': 72, 'reentrancy': 58, None: 47, 'time_manipulation': 45, 'unchecked_low_level_calls': 44, 'denial_of_service': 6})


In [7]:
bad = [r for r in forced_results if r["status"] == "parse_error"]
print(len(bad))
for r in bad[:5]:
    print("---", r["source"], os.path.basename(r["path"])[-45:])
    print(repr(r["raw_text"][:250]))

47
--- smartbugs spank_chain_payment.sol
'I need to analyze this Solidity contract for security vulnerabilities across the five categories.\n\nLet me examine the code systematically:\n\n1. **Reentrancy**: Looking for external calls before state updates:\n   - Line 242: `_spender.call(...)` in `ap'
--- smartbugs x663e4229142a27f00bafb5d087e1e730648314c3.sol
'I need to analyze this Solidity contract for security vulnerabilities across the five categories.\n\nLet me examine the code systematically:\n\n1. **Reentrancy**: Looking for external calls before state updates.\n   - Line 1136: `msg.sender.send(autoBirth'
--- smartbugs smart_billions.sol
'I need to analyze this Solidity contract for security vulnerabilities across the five categories.\n\nLet me examine the key functions:\n\n1. **Reentrancy**: Looking at external calls and state updates:\n   - Line 253: `msg.sender.transfer(_amount)` in `co'
--- dappscan 44b76f3056f0ed7229db8c__contracts__CToken.sol
'I need to analyze this Solidity 